# DataMatrix 코드 판독 가능성 판별 모델

촬영된 DM코드 이미지를 보고 **판독 성공 가능 / 실패**를 미리 판정하는 AI를 만듭니다.

- 데이터: `qr_crops`(성공) / `qr_crops_failed`(실패)
- 방식: ResNet18 전이학습 (ImageNet 사전학습 모델을 우리 데이터로 미세조정)
- 실행: 위에서부터 셀을 하나씩 `Shift+Enter`

## 1. 환경 확인

In [ ]:
import torch, torchvision
print("PyTorch:", torch.__version__)
print("GPU 사용:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("장비:", torch.cuda.get_device_name(0))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 2. 데이터 확인

In [ ]:
from pathlib import Path

OK_DIR = Path("qr_crops")
NG_DIR = Path("qr_crops_failed")

ok_files = sorted(OK_DIR.glob("*.jpg"))
ng_files = sorted(NG_DIR.glob("*.jpg"))

print(f"판독 성공: {len(ok_files):,}장")
print(f"판독 실패: {len(ng_files):,}장")
print(f"합계    : {len(ok_files)+len(ng_files):,}장")

if len(ok_files) == 0 or len(ng_files) == 0:
    print("\n[확인 필요] 이미지가 0장입니다. 아래로 현재 위치를 확인하세요:")
    import os; print(os.getcwd(), os.listdir("."))

## 3. 샘플 이미지 보기\n위: 판독 성공 / 아래: 판독 실패 — 눈으로 차이를 확인합니다.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

fig, axes = plt.subplots(2, 6, figsize=(15, 5.5))
for i in range(6):
    axes[0, i].imshow(Image.open(ok_files[i])); axes[0, i].axis("off")
    axes[0, i].set_title("OK", color="green", fontsize=11)
    axes[1, i].imshow(Image.open(ng_files[i])); axes[1, i].axis("off")
    axes[1, i].set_title("FAIL", color="red", fontsize=11)
plt.tight_layout(); plt.show()

## 4. 데이터셋 준비\n이미지 크기를 128×128로 통일하고, 학습용 80% / 검증용 20%로 나눕니다.

In [ ]:
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms

IMG_SIZE = 128
BATCH = 64

train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

class DMDataset(Dataset):
    def __init__(self, files, labels, tf):
        self.files, self.labels, self.tf = files, labels, tf
    def __len__(self):
        return len(self.files)
    def __getitem__(self, i):
        img = Image.open(self.files[i]).convert("RGB")
        return self.tf(img), self.labels[i]

all_files  = ok_files + ng_files
all_labels = [0] * len(ok_files) + [1] * len(ng_files)   # 0=성공, 1=실패

idx = list(range(len(all_files)))
gen = torch.Generator().manual_seed(42)
n_val = int(len(idx) * 0.2)
val_idx, train_idx = random_split(idx, [n_val, len(idx) - n_val], generator=gen)

train_ds = DMDataset([all_files[i] for i in train_idx], [all_labels[i] for i in train_idx], train_tf)
val_ds   = DMDataset([all_files[i] for i in val_idx],   [all_labels[i] for i in val_idx],   eval_tf)

train_dl = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=2)
val_dl   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=2)

print(f"학습용: {len(train_ds):,}장 / 검증용: {len(val_ds):,}장")

## 5. 모델 준비 (ResNet18 전이학습)

In [ ]:
import torch.nn as nn
from torchvision.models import resnet18, ResNet18_Weights

model = resnet18(weights=ResNet18_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features, 2)   # 2가지로 분류: 성공/실패
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
print("모델 준비 완료 —", sum(p.numel() for p in model.parameters())/1e6, "백만 파라미터")

## 6. 학습\nGPU에서 5에폭 학습합니다 (약 2~4분). 에폭마다 검증 정확도가 출력됩니다.

In [ ]:
import time

EPOCHS = 5
best_acc = 0.0

for ep in range(1, EPOCHS + 1):
    t0 = time.time()
    model.train()
    tr_loss = 0.0
    for x, y in train_dl:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()
        tr_loss += loss.item() * x.size(0)

    model.eval()
    correct = total = 0
    with torch.no_grad():
        for x, y in val_dl:
            x, y = x.to(device), y.to(device)
            pred = model(x).argmax(1)
            correct += (pred == y).sum().item()
            total += y.size(0)
    acc = correct / total
    print(f"[{ep}/{EPOCHS}] 학습손실 {tr_loss/len(train_ds):.4f} | 검증 정확도 {acc*100:.2f}% | {time.time()-t0:.0f}초")

    if acc > best_acc:
        best_acc = acc
        torch.save(model.state_dict(), "dm_model_best.pt")

print(f"\n최고 검증 정확도: {best_acc*100:.2f}%  (dm_model_best.pt 저장됨)")

## 7. 결과 분석 — 혼동행렬\n어떤 걸 맞히고 어떤 걸 틀렸는지 봅니다.

In [ ]:
import numpy as np

model.load_state_dict(torch.load("dm_model_best.pt"))
model.eval()

cm = np.zeros((2, 2), dtype=int)
with torch.no_grad():
    for x, y in val_dl:
        p = model(x.to(device)).argmax(1).cpu()
        for t, pr in zip(y, p):
            cm[t, pr] += 1

labels = ["성공(OK)", "실패(FAIL)"]
print("            예측:성공   예측:실패")
for i, lab in enumerate(labels):
    print(f"실제 {lab:10} {cm[i,0]:6d} {cm[i,1]:9d}")

acc = cm.diagonal().sum() / cm.sum()
recall_ng = cm[1,1] / cm[1].sum()
print(f"\n전체 정확도: {acc*100:.2f}%")
print(f"불량(실패) 검출률: {recall_ng*100:.2f}%  ← 현장에서 중요한 지표")

## 8. 실제 판정 해보기\n검증 데이터에서 임의로 골라 모델 판정과 정답을 비교합니다.

In [ ]:
import random

model.eval()
picks = random.sample(range(len(val_ds)), 8)
fig, axes = plt.subplots(2, 4, figsize=(13, 6))

for ax, i in zip(axes.flat, picks):
    x, y = val_ds[i]
    with torch.no_grad():
        prob = torch.softmax(model(x.unsqueeze(0).to(device)), 1)[0]
    pred = int(prob.argmax())
    img = Image.open(val_ds.files[i]).convert("RGB")
    ax.imshow(img); ax.axis("off")
    name = ["OK", "FAIL"][pred]
    ok = (pred == y)
    ax.set_title(f"AI판정: {name} ({prob[pred]*100:.0f}%)\n정답: {['OK','FAIL'][y]}",
                 color=("green" if ok else "red"), fontsize=10)
plt.tight_layout(); plt.show()

## 9. 다음 단계

- **정확도를 더 올리려면**: 위 6번 셀의 `EPOCHS`를 10~20으로 늘려 다시 실행
- **모델 파일**: `dm_model_best.pt` (볼륨에 저장되어 노트북을 껐다 켜도 남아 있음)
- **서비스로 배포**: 이 모델을 포털의 Serving 기능으로 올리면 API로 호출 가능
  (배포용 변환이 필요하니 준비되면 알려주세요)